# 🔄 Stock Bot AutoEncoder Fine-tuning on Google Colab

이 노트북은 기존에 훈련된 AutoEncoder 모델을 새로운 데이터로 Fine-tuning합니다.

## 📋 Overview
- **목적**: 기존 모델을 새로운 데이터에 적응시키기
- **방법**: 낮은 학습률로 점진적 학습
- **데이터**: 새로운 월의 전처리된 HDF5 배치 파일
- **훈련 시간**: 약 30분-1시간 (기존 모델 기반)

## 🎯 Fine-tuning 전략
1. **기존 모델 로드**: 사전 훈련된 체크포인트 사용
2. **낮은 학습률**: 기존 지식 보존
3. **적은 에포크**: 과적합 방지
4. **점진적 적응**: 새로운 패턴 학습

## 🔧 Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install h5py numpy pandas tqdm matplotlib seaborn

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup directories
import os
os.chdir('/content')
!mkdir -p models logs data

# Data path
DATA_PATH = '/content/drive/MyDrive/ColabData/models/stockbot/pre_training_data'
MODEL_PATH = '/content/drive/MyDrive/ColabData/models/stockbot/autoencoder/20251011_154401'  # 사전 훈련된 모델

# Check data
if os.path.exists(DATA_PATH):
    print(f"✅ Data found at: {DATA_PATH}")
    months = [d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))]
    print(f"Available months: {sorted(months)}")
else:
    print(f"❌ Data not found at: {DATA_PATH}")

# Check existing models
if os.path.exists(MODEL_PATH):
    print(f"✅ Models found at: {MODEL_PATH}")
    models = [d for d in os.listdir(MODEL_PATH) if os.path.isdir(os.path.join(MODEL_PATH, d))]
    print(f"Available models: {sorted(models)}")
else:
    print(f"❌ Models not found at: {MODEL_PATH}")

In [ ]:
# GitHub 저장소 클론 (프로젝트 코드)
import os
from google.colab import userdata

# 🔑 아이콘 클릭 → Add new secret
# Name: GITHUB_TOKEN
# Value: your_personal_access_token
try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token for authentication")
except:
    print("⚠️ GITHUB_TOKEN not found. Using public clone (may have rate limits)")
    use_token = False

# 이미 클론되어 있으면 스킵
if not os.path.exists('/content/stock-bot2'):
    print("📥 Cloning repository...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git /content/stock-bot2
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git /content/stock-bot2
else:
    print("📁 Repository already exists, updating...")
    %cd /content/stock-bot2
    !git fetch origin && git pull

# 작업 디렉토리 변경
%cd /content/stock-bot2

# Python path 추가
import sys
sys.path.append('/content/stock-bot2')

print("✅ Repository cloned and ready!")
print(f"📂 Current directory: {os.getcwd()}")

## 🎯 Model Selection and Configuration

In [ ]:
# 사용 가능한 모델 목록 표시
print("📋 Available pre-trained models:")
models = [d for d in os.listdir(MODEL_PATH) if os.path.isdir(os.path.join(MODEL_PATH, d))]
for i, model in enumerate(sorted(models)):
    model_info_path = os.path.join(MODEL_PATH, model, 'model_info.json')
    if os.path.exists(model_info_path):
        import json
        with open(model_info_path, 'r') as f:
            info = json.load(f)
        print(f"  {i}: {model}")
        print(f"     - Best Val Loss: {info.get('best_val_loss', 'N/A'):.6f}")
        print(f"     - Parameters: {info.get('total_params', 'N/A'):,}")
        print(f"     - Training Time: {info.get('training_time_minutes', 'N/A'):.1f} min")
        print(f"     - Data Shape: {info.get('data_shape', {}).get('seq_len', 'N/A')} x {info.get('data_shape', {}).get('num_features', 'N/A')}")
    else:
        print(f"  {i}: {model} (no info available)")

# 모델 선택 (가장 최신 모델을 기본으로)
if models:
    selected_model = sorted(models)[-1]  # 가장 최신 모델
    print(f"
🎯 Selected model: {selected_model}")
    
    # 다른 모델을 선택하려면 아래 라인의 주석을 해제하고 인덱스를 변경하세요
    # selected_model = sorted(models)[0]  # 인덱스 변경
    
    PRETRAINED_MODEL_PATH = os.path.join(MODEL_PATH, selected_model, 'model.pt')
    print(f"📁 Model path: {PRETRAINED_MODEL_PATH}")
else:
    print("❌ No pre-trained models found!")
    PRETRAINED_MODEL_PATH = None

In [ ]:
# Fine-tuning Configuration
FINETUNE_CONFIG = {
    # 데이터 설정
    'train_months': ['2025_01', '2025_02'],  # 새로운 훈련 데이터 (수정 필요)
    'val_months': ['2025_03'],               # 새로운 검증 데이터 (수정 필요)
    
    # 훈련 설정 (Fine-tuning용으로 조정)
    'max_epochs': 5,                         # 적은 에포크
    'learning_rate': 1e-4,                   # 낮은 학습률
    'weight_decay': 1e-5,                    # 낮은 weight decay
    'batch_size': 2,                         # 작은 배치 크기
    
    # Fine-tuning 전략
    'freeze_encoder': False,                 # 인코더 동결 여부
    'freeze_layers': 0,                      # 동결할 레이어 수
    'warmup_epochs': 1,                      # 워밍업 에포크
    'lr_schedule': 'cosine',                 # 학습률 스케줄
    
    # 정규화
    'dropout_increase': 0.05,                # 드롭아웃 증가
    'gradient_clip': 0.5,                    # 그래디언트 클리핑
}

# GPU 메모리에 따른 조정
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️ GPU Memory: {gpu_memory:.1f} GB")
    
    if gpu_memory < 12:
        FINETUNE_CONFIG['batch_size'] = 1
        print("⚠️ Adjusted config for limited GPU memory")
    elif gpu_memory > 25:
        FINETUNE_CONFIG['batch_size'] = 4
        print("🚀 Adjusted config for high-end GPU")

print("
📋 Fine-tuning Configuration:")
for key, value in FINETUNE_CONFIG.items():
    print(f"  {key}: {value}")

## 🚀 CLI-based Fine-tuning

In [ ]:
# CLI를 사용한 Fine-tuning 실행
print("🚀 Starting CLI-based fine-tuning...")

# 출력 디렉토리 설정
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = f'/content/drive/MyDrive/stock_bot_models/finetuned_{timestamp}'

# 출력 디렉토리 생성
os.makedirs(output_dir, exist_ok=True)
print(f"📁 Output directory created: {output_dir}")

# CLI 명령어 구성
cmd = f"""python scripts/fine_tuning/finetune_autoencoder.py \\
    --model "{PRETRAINED_MODEL_PATH}" \\
    --data "{DATA_PATH}" \\
    --output "{output_dir}" \\
    --train-months {' '.join(FINETUNE_CONFIG['train_months'])} \\
    --val-months {' '.join(FINETUNE_CONFIG['val_months'])} \\
    --learning-rate {FINETUNE_CONFIG['learning_rate']} \\
    --max-epochs {FINETUNE_CONFIG['max_epochs']} \\
    --batch-size {FINETUNE_CONFIG['batch_size']} \\
    --weight-decay {FINETUNE_CONFIG['weight_decay']} \\
    --freeze-layers {FINETUNE_CONFIG['freeze_layers']} \\
    --warmup-epochs {FINETUNE_CONFIG['warmup_epochs']} \\
    --lr-schedule {FINETUNE_CONFIG['lr_schedule']} \\
    --dropout-increase {FINETUNE_CONFIG['dropout_increase']} \\
    --gradient-clip {FINETUNE_CONFIG['gradient_clip']} \\
    --device cuda \\
    --seed 42"""

# 인코더 동결 옵션 추가
if FINETUNE_CONFIG['freeze_encoder']:
    cmd += " --freeze-encoder"

print("📝 Fine-tuning command:")
print(cmd)
print("⏳ Executing fine-tuning...")

# 실행
!{cmd}

## 📊 Results Analysis

In [ ]:
# 결과 분석
import json
import matplotlib.pyplot as plt

# 모델 정보 로드
model_info_path = os.path.join(output_dir, 'model_info.json')
if os.path.exists(model_info_path):
    with open(model_info_path, 'r') as f:
        model_info = json.load(f)
    
    print("📊 Fine-tuning Results:")
    print(f"  Original Best Loss: {model_info['original_best_loss']:.6f}")
    print(f"  Initial Loss on New Data: {model_info['initial_val_loss']:.6f}")
    print(f"  Fine-tuned Best Loss: {model_info['best_val_loss']:.6f}")
    print(f"  Improvement: {model_info['improvement']:.6f} ({model_info['improvement_percent']:.2f}%)")
    print(f"  Training Time: {model_info['training_time_minutes']:.1f} minutes")
    print(f"  Epochs: {model_info['epochs']}")
    
    # 훈련 히스토리 로드
    history_path = os.path.join(output_dir, 'training_history.json')
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
        
        finetune_history = history['finetune_history']
        
        # 결과 시각화
        plt.figure(figsize=(15, 5))
        
        # 1. 손실 곡선
        plt.subplot(1, 3, 1)
        epochs = range(1, len(finetune_history['train_loss']) + 1)
        plt.plot(epochs, finetune_history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        plt.plot(epochs, finetune_history['val_loss'], 'r-', label='Val Loss', linewidth=2)
        plt.axhline(y=model_info['initial_val_loss'], color='orange', linestyle='--', label='Initial Val Loss')
        plt.axhline(y=model_info['original_best_loss'], color='green', linestyle='--', label='Original Best')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Fine-tuning Loss Curves')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 2. 학습률 변화
        plt.subplot(1, 3, 2)
        plt.plot(epochs, finetune_history['learning_rate'], 'g-', linewidth=2)
        plt.xlabel('Epoch')
        plt.ylabel('Learning Rate')
        plt.title('Learning Rate Schedule')
        plt.grid(True, alpha=0.3)
        plt.yscale('log')
        
        # 3. 비교 막대 그래프
        plt.subplot(1, 3, 3)
        models = ['Original\
Best', 'Initial on\
New Data', 'Fine-tuned\
Best']
        losses = [model_info['original_best_loss'], model_info['initial_val_loss'], model_info['best_val_loss']]
        colors = ['green', 'orange', 'blue']
        bars = plt.bar(models, losses, color=colors, alpha=0.7)
        plt.ylabel('Validation Loss')
        plt.title('Model Comparison')
        plt.xticks(rotation=45)
        
        # 막대 위에 값 표시
        for bar, loss in zip(bars, losses):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                     f'{loss:.4f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('/content/finetuning_results.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("📊 Results visualization saved as 'finetuning_results.png'")
    else:
        print("❌ Training history not found")
else:
    print("❌ Model info not found. Check if fine-tuning completed successfully.")

## 🧪 Test Fine-tuned Model

In [ ]:
# Fine-tuned 모델 테스트
print("🧪 Testing fine-tuned model...")

# 모델 로드
model_path = os.path.join(output_dir, 'model.pt')
if os.path.exists(model_path):
    checkpoint = torch.load(model_path, map_location='cuda')
    
    # 모델 설정 로드
    original_config = checkpoint['original_config']
    
    # 모델 생성
    from ai_trader.embedding.autoencoder_model import MaskedAutoEncoder
    
    model = MaskedAutoEncoder(
        input_dim=original_config.get('num_features', 28),
        embedding_dim=original_config['embedding_dim'],
        hidden_dim=original_config['hidden_dim'],
        seq_len=original_config.get('seq_len', 60),
        num_layers=original_config['num_layers'],
        dropout=original_config['dropout'],
        mask_ratio=original_config['mask_ratio']
    ).cuda()
    
    # 가중치 로드
    model.load_state_dict(checkpoint['model_state_dict'])
    
    print("✅ Fine-tuned model loaded successfully!")
    
    # 추론 테스트
    model.eval()
    with torch.no_grad():
        # 샘플 데이터 생성
        sample_input = torch.randn(1, original_config.get('seq_len', 60), original_config.get('num_features', 28)).cuda()
        
        # 추론
        reconstruction, embedding, mask = model(sample_input)
        
        print(f"🔍 Inference Test Results:")
        print(f"  Input shape: {sample_input.shape}")
        print(f"  Embedding shape: {embedding.shape}")
        print(f"  Reconstruction shape: {reconstruction.shape}")
        print(f"  Mask ratio: {mask.float().mean():.3f}")
        
        # 재구성 품질
        recon_loss = torch.nn.functional.mse_loss(reconstruction[mask], sample_input[mask])
        print(f"  Reconstruction loss: {recon_loss.item():.6f}")
        
        # 임베딩 통계
        print(f"  Embedding mean: {embedding.mean().item():.4f}")
        print(f"  Embedding std: {embedding.std().item():.4f}")
        print(f"  Embedding norm: {torch.norm(embedding, dim=1).mean().item():.4f}")
    
    print("✅ Model inference test completed!")
else:
    print("❌ Fine-tuned model not found. Check if fine-tuning completed successfully.")

## 🎯 Conclusion

이 노트북에서는 CLI 도구를 사용하여 기존에 훈련된 AutoEncoder 모델을 새로운 데이터로 Fine-tuning했습니다.

### 🔑 Key Points

1. **효율적인 적응**: 기존 모델의 지식을 보존하면서 새로운 패턴 학습
2. **빠른 훈련**: 전체 재훈련 대비 훨씬 짧은 시간
3. **성능 개선**: 새로운 데이터에 대한 성능 향상
4. **CLI 기반**: 간단하고 재현 가능한 워크플로우

### 📈 Benefits

- **시간 절약**: 전체 재훈련 대비 80% 이상 시간 단축
- **성능 유지**: 기존 성능을 유지하면서 새로운 데이터 적응
- **메모리 효율**: 작은 배치 크기로 메모리 사용량 최적화
- **점진적 학습**: 새로운 월별 데이터에 대한 지속적인 적응

### 🚀 Next Steps

1. **GRPO 훈련**: Fine-tuned embedding을 사용한 강화학습 에이전트 훈련
2. **성능 평가**: 실제 거래 시뮬레이션을 통한 성능 검증
3. **지속적 개선**: 새로운 데이터가 추가될 때마다 Fine-tuning 반복
4. **하이퍼파라미터 최적화**: 더 나은 Fine-tuning 전략 탐색

### 💡 CLI Command Reference

```bash
# 기본 Fine-tuning
python scripts/fine_tuning/finetune_autoencoder.py \\
    --model models/pretrained/model.pt \\
    --data data/preprocessed \\
    --output models/finetuned \\
    --train-months 2025_01 2025_02 \\
    --val-months 2025_03 \\
    --learning-rate 1e-4 \\
    --max-epochs 5

# Conservative Fine-tuning (더 안전한 설정)
python scripts/fine_tuning/finetune_autoencoder.py \\
    --model models/pretrained/model.pt \\
    --data data/preprocessed \\
    --output models/finetuned \\
    --train-months 2025_01 2025_02 \\
    --val-months 2025_03 \\
    --learning-rate 5e-5 \\
    --max-epochs 3 \\
    --freeze-layers 2 \\
    --dropout-increase 0.02

# Aggressive Fine-tuning (더 적극적인 설정)
python scripts/fine_tuning/finetune_autoencoder.py \\
    --model models/pretrained/model.pt \\
    --data data/preprocessed \\
    --output models/finetuned \\
    --train-months 2025_01 2025_02 \\
    --val-months 2025_03 \\
    --learning-rate 2e-4 \\
    --max-epochs 8 \\
    --freeze-layers 0 \\
    --dropout-increase 0.1
```